## Database: SQL Server

In [23]:
# ====================================================
# Installing pyodbc in Jupyter Notebook
# ====================================================

# !pip install pyodbc
# -------------------
# This command tells Python to download and install the 'pyodbc' library.
# In Jupyter, the ! at the beginning allows you to run shell commands.
# Shell commands are the same as running commands in your terminal/command prompt.

# pyodbc is required to:
# - Connect Python to SQL Server
# - Execute SQL queries (SELECT, INSERT, UPDATE, DELETE)
# - Fetch data from the database into Python

# Run this cell ONCE before trying to use 'import pyodbc'
#!pip install pyodbc

In [24]:
# ====================================================
# Importing pyodbc 
# ====================================================

# pyodbc is a Python library that allows Python programs
# to connect to databases using ODBC drivers.

# In our case, we use pyodbc to:
# - Connect Python to Microsoft SQL Server
# - Run SQL commands (SELECT, INSERT, UPDATE, DELETE)
# - Read results from the database

# Without pyodbc, Python cannot talk to SQL Server.

import pyodbc

In [9]:
# ====================================================
# SQL Server Connection String 
# ====================================================

# This connection string tells Python HOW to connect to SQL Server.
# It is used by pyodbc (or similar libraries).

conn_str = (
    # 1. DRIVER
    # This specifies which ODBC driver to use.
    # "ODBC Driver 17 for SQL Server" must be installed on your machine.
    'DRIVER={ODBC Driver 17 for SQL Server};'

    # 2. SERVER
    # The SQL Server instance name.
    # "localhost" means the database is on the same machine.
    # "SQLEXPRESS" is the default name for SQL Server Express edition.
    'SERVER=localhost\\SQLEXPRESS;'

    # 3. DATABASE
    # The name of the database you want to connect to.
    'DATABASE=Company;'

    # 4. AUTHENTICATION
    # Trusted_Connection=yes means:
    # - Use Windows Authentication
    # - No username or password is required
    # - The current Windows user must have DB access
    'Trusted_Connection=yes;'
)

In [10]:
# ====================================================
# Database Connection Function
# ====================================================

def get_db_connection():
    """
    This function creates and returns a NEW connection
    to the SQL Server database.

    Why we use a function:
    - Avoid repeating connection code everywhere
    - Easier to maintain and update
    - Cleaner and more professional code
    """

    # 1. Use pyodbc to open a connection to SQL Server
    # The 'conn_str' contains:
    # - Driver
    # - Server name
    # - Database name
    # - Authentication method
    connection = pyodbc.connect(conn_str)

    # 2. Return the open connection
    # The calling code will:
    # - Create a cursor
    # - Execute SQL commands
    # - Close the connection when finished
    return connection

In [11]:
# ====================================================
# Function to Add a Customer to SQL Server
# ====================================================

def add_customer(customer):
    """
    This function adds ONE customer to the SQL Server database.

    Parameters:
    -----------
    customer : dict
        - A Python dictionary (object) that contains all the customer information.
        - Example:
            {
                "customer_id": "C001",
                "name": "Ahmed",
                "age": 30,
                "gender": "Male",
                "region": "Cairo"
            }

    Returns:
    --------
    tuple (success, message)
        - success : True if added successfully, False if an error occurred
        - message : a string explaining the result
    """

    try:
        # 1. Open a connection to the database
        #    - get_db_connection() returns a pyodbc connection object
        #    - Think of this as “opening the door” to the database
        conn = get_db_connection()

        # 2. Create a cursor
        #    - A cursor is like a “pointer” that allows Python to run SQL commands
        cursor = conn.cursor()

        # 3. Write the SQL INSERT command
        #    - ? symbols are placeholders for values
        #    - Using placeholders prevents SQL injection attacks (safer)
        sql_insert = """
            INSERT INTO dbo.Customers (CustomerID, Name, Age, Gender, Region)
            VALUES (?, ?, ?, ?, ?)
        """

        # 4. Execute the SQL command with the data from the 'customer' dictionary
        cursor.execute(
            sql_insert,
            customer['customer_id'],  # Customer unique ID
            customer['name'],         # Customer name
            customer['age'],          # Customer age
            customer['gender'],       # Customer gender
            customer['region']        # Customer region
        )

        # 5. Save the changes in the database
        #    - commit() tells the database: “Yes, save everything I just did”
        conn.commit()

        # 6. Return a success message
        return True, "Customer added successfully"

    except Exception as e:
        # 7. If anything goes wrong (wrong data, connection error, etc.)
        #    - Catch the error and return False with the error message
        return False, str(e)

    finally:
        # 8. ALWAYS close the database connection
        #    - This runs no matter what
        #    - Think of it as “closing the door” after finishing your work
        conn.close()

In [12]:
# ====================================================
# Example: Add a Customer to the Database
# ====================================================

# 1. Create a customer object
# - In Python, this is a dictionary (like a JSON object)
# - Each key represents a field/column in the database
# - Each value is the data for that customer
new_customer = {
    "customer_id": 1,     # Unique ID for this customer
    "name": "Ahmed Fahmy",     # Customer's name
    "age": 35,                 # Customer's age
    "gender": "Male",          # Customer's gender
    "region": "Cairo"          # Customer's region
}

# 2. Call the add_customer() function
# - Pass the customer dictionary we just created
# - The function will handle connecting to the database and inserting the record
success, message = add_customer(new_customer)

# 3. Show the result to the user
# - success: True if the customer was added successfully, False if there was an error
# - message: A string explaining the result (success or error message)
print("Success:", success)
print("Message:", message)

Success: True
Message: Customer added successfully


In [13]:
# ====================================================
# Function to Get All Customers from Database
# ====================================================

def get_customers():
    """
    This function reads ALL customers from the database and returns them as a list of dictionaries.

    Steps it performs:
    1. Connects to the SQL Server database
    2. Executes a SELECT query to get all customer records
    3. Converts each database row into a Python dictionary (object)
    4. Returns a list of customer dictionaries
    """

    try:
        # 1. Open a connection to the database
        # - get_db_connection() opens the "door" to SQL Server
        conn = get_db_connection()

        # 2. Create a cursor
        # - A cursor allows us to run SQL commands on the database
        cursor = conn.cursor()

        # 3. Write and execute the SQL SELECT query
        # - This query retrieves all columns for all customers
        cursor.execute("""
            SELECT CustomerID, Name, Age, Gender, Region
            FROM dbo.Customers
        """)

        # 4. Fetch all rows returned by the query
        # - rows is a list of records from the database
        rows = cursor.fetchall()

        # 5. Create an empty list to store customer dictionaries
        customers = []

        # 6. Loop through each row and convert it into a Python dictionary
        for row in rows:
            customer = {
                "customer_id": row.CustomerID,  # Get CustomerID column
                "name": row.Name,               # Get Name column
                "age": row.Age,                 # Get Age column
                "gender": row.Gender,           # Get Gender column
                "region": row.Region            # Get Region column
            }
            # Add the dictionary to the list of customers
            customers.append(customer)

        # 7. Return True to indicate success, and the list of customer dictionaries
        return True, customers

    except Exception as e:
        # 8. If something goes wrong (e.g., connection error), return False and the error message
        return False, str(e)

    finally:
        # 9. ALWAYS close the database connection
        # - This ensures we don’t leave connections open, even if an error occurs
        conn.close()

In [14]:
# ====================================================
# Example: Get All Customers from Database
# ====================================================

# 1. Call the function to get all customers
# - get_customers() returns two things:
#   1. success → True if the function worked, False if there was an error
#   2. customers → list of customer dictionaries if success=True, or error message if success=False
success, customers = get_customers()

# 2. Check if the function was successful
if success:
    # ✅ Function worked, we have a list of customers
    print("Customers found:", len(customers))  # Show total number of customers

    # Loop through each customer and print its details
    for c in customers:
        # c is a dictionary representing one customer
        print(c)

else:
    # ❌ Something went wrong (e.g., database connection error)
    # - customers variable contains the error message
    print("Error:", customers)

Customers found: 1
{'customer_id': 1, 'name': 'Ahmed Fahmy', 'age': 35, 'gender': 'Male', 'region': 'Cairo'}


In [15]:
# ====================================================
# Function to Update an Existing Customer
# ====================================================

def update_customer(customer_id, customer):
    """
    This function updates an existing customer in the database.

    Parameters:
    -----------
    customer_id : str
        - The ID of the customer we want to update

    customer : dict
        - A Python dictionary containing the NEW customer data
        - Example:
          {
              "name": "Ahmed Ali",
              "age": 31,
              "gender": "Male",
              "region": "Giza"
          }

    Returns:
    --------
    tuple (success, message)
        - success : True if update was successful, False if an error occurred
        - message : string explaining the result or error
    """

    try:
        # 1. Open a connection to the database
        # - Think of this as “opening the door” to the database
        conn = get_db_connection()

        # 2. Create a cursor
        # - A cursor allows Python to run SQL commands
        cursor = conn.cursor()

        # 3. Prepare and execute the SQL UPDATE command
        # - The ? symbols are placeholders for values (prevents SQL injection)
        cursor.execute("""
            UPDATE dbo.Customers
            SET Name = ?, Age = ?, Gender = ?, Region = ?
            WHERE CustomerID = ?
        """,
        customer['name'],     # New name
        customer['age'],      # New age
        customer['gender'],   # New gender
        customer['region'],   # New region
        customer_id)          # Which customer to update

        # 4. Save changes to the database
        # - commit() tells the database: “Yes, save these changes permanently”
        conn.commit()

        # 5. Check if any rows were affected
        # - If rowcount is 0, the customer ID does not exist
        if cursor.rowcount == 0:
            return False, "Customer not found"

        # 6. Return success message
        return True, "Customer updated successfully"

    except Exception as e:
        # 7. If anything goes wrong (connection error, wrong data, etc.)
        # - Return False and the error message
        return False, str(e)

    finally:
        # 8. ALWAYS close the database connection
        # - This ensures we do not leave the connection open, even if an error occurs
        conn.close()

In [16]:
# ====================================================
# Example: Update an Existing Customer
# ====================================================

# 1. Create a dictionary with the updated customer information
# - This is the new data we want to save for the customer
updated_customer = {
    "name": "Ahmed Fahmy Ali",  # Updated full name
    "age": 36,                  # Updated age
    "gender": "Male",           # Gender remains the same
    "region": "Giza"            # Updated region
}

# 2. Specify the Customer ID of the customer we want to update
customer_id = 1

# 3. Call the update_customer() function
# - Pass the customer ID and the new customer dictionary
# - The function will handle connecting to the database and updating the record
success, message = update_customer(customer_id, updated_customer)

# 4. Show the result
# - success: True if update worked, False if something went wrong
# - message: Explanation of what happened (success or error message)
print("Success:", success)
print("Message:", message)

Success: True
Message: Customer updated successfully


In [18]:
# ====================================================
# Example: Verify Updated Customer Data
# ====================================================

# 1. Fetch all customers from the database
# - get_customers() returns a tuple: (success flag, list of customers or error message)
success, customers = get_customers()

# 2. Check if fetching customers was successful
if success:
    # ✅ Loop through each customer in the list
    for c in customers:
        # 3. Find the customer with the updated ID
        if c["customer_id"] == 1:
            # 4. Print the updated customer data
            print("Updated Customer:", c)
else:
    # ❌ If fetching failed, print the error message
    print("Error fetching customers:", customers)

Updated Customer: {'customer_id': 1, 'name': 'Ahmed Fahmy Ali', 'age': 36, 'gender': 'Male', 'region': 'Giza'}


In [19]:
# ====================================================
# Function to Delete a Customer from Database
# ====================================================

def delete_customer(customer_id):
    """
    This function deletes ONE customer from the SQL Server database.

    Parameters:
    -----------
    customer_id : str
        - The ID of the customer we want to delete

    Returns:
    --------
    tuple (success, message)
        - success : True if deletion was successful, False if an error occurred
        - message : Explanation of the result or error
    """

    try:
        # 1. Open a connection to the database
        # - Think of this as "opening the door" to SQL Server
        conn = get_db_connection()

        # 2. Create a cursor
        # - A cursor allows Python to run SQL commands
        cursor = conn.cursor()

        # 3. Prepare and execute the DELETE SQL command
        # - ? is a placeholder to safely pass the customer ID
        cursor.execute("""
            DELETE FROM dbo.Customers
            WHERE CustomerID = ?
        """, customer_id)

        # 4. Save changes to the database
        conn.commit()

        # 5. Check if any rows were affected
        # - If rowcount is 0, the customer ID does not exist
        if cursor.rowcount == 0:
            return False, "Customer not found"

        # 6. Return success message
        return True, "Customer deleted successfully"

    except Exception as e:
        # 7. If anything goes wrong (e.g., connection error, SQL error)
        return False, str(e)

    finally:
        # 8. ALWAYS close the database connection
        # - Ensures no open connections remain even if an error occurs
        conn.close()

In [20]:
# ====================================================
# Example: Delete a Customer from Database
# ====================================================

# 1. Specify the Customer ID we want to delete
# - This should match the customer ID in the database
customer_id = 1

# 2. Call the delete_customer() function
# - Pass the customer ID to the function
# - The function will handle connecting to the database and deleting the record
success, message = delete_customer(customer_id)

# 3. Show the result
# - success: True if deletion worked, False if there was an error
# - message: Explanation of what happened (success or error message)
print("Success:", success)
print("Message:", message)

Success: True
Message: Customer deleted successfully


In [21]:
# ====================================================
# Example: Verify Customer Deletion
# ====================================================

# 1. Fetch all customers from the database
# - get_customers() returns two things:
#   1. success → True if the function worked, False if there was an error
#   2. customers → list of customer dictionaries if success=True, or error message if success=False
success, customers = get_customers()

# 2. Check if fetching customers was successful
if success:
    # ✅ Show the total number of remaining customers
    print("Remaining customers:", len(customers))

    # Loop through each remaining customer and print its details
    for c in customers:
        print(c)
else:
    # ❌ If fetching failed, print the error message
    print("Error fetching customers:", customers)

Remaining customers: 0
